# Week 1 · BitNet b1.58 — Ternary Weights from Scratch

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week1_quantization.md`](../docs/theory/week1_quantization.md)

---

### Learning objectives

By the end of this notebook you will be able to

1. Derive **absmax** and **zero-point** quantization from first principles and state their numerical-error bounds.
2. Explain why **Quantization-Aware Training (QAT)** dominates **Post-Training Quantization (PTQ)** in the ≤4-bit regime.
3. Implement the **Straight-Through Estimator (STE)** in three lines of PyTorch and prove gradient flow.
4. Build the **BitNet b1.58** ternarization rule and a working `BitLinear` layer.
5. Quantify the disk-size and accuracy trade-off vs. an FP32 baseline on a controlled task.


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Mathematical derivation

### 1.1 Absmax (symmetric) quantization

Given a tensor $x\in\mathbb{R}^n$ and a target bit-width $b$, the symmetric absmax quantizer is

$$
s = \frac{2^{b-1}-1}{\max_i |x_i|}, \qquad q_i = \operatorname{round}(s\cdot x_i), \qquad \tilde x_i = q_i/s.
$$

The worst-case quantization error per element is bounded by $1/(2s)$, so error decays **linearly** with the number of bits.

### 1.2 Zero-point (asymmetric) quantization

For activations after a ReLU/GELU the distribution is skewed; an offset $z$ recovers a few bits of headroom:

$$
s = \frac{\max(x)-\min(x)}{2^b-1}, \quad
z = -\operatorname{round}\!\left(\frac{\min(x)}{s}\right) - 2^{b-1}, \quad
q_i = \operatorname{round}(x_i/s) + z.
$$

### 1.3 BitNet b1.58 ternarization

The b1.58 variant constrains weights to $W_{ij}\in\{-1,0,+1\}$, encoded in $\log_2 3 \approx 1.58$ bits per weight. The ternarization rule uses the **mean absolute weight** as the per-tensor scale:

$$
\gamma = \frac{1}{nm}\sum_{i,j}|W_{ij}|, \qquad
W_{\text{tern}} = \operatorname{round}\!\left(\operatorname{clip}\!\left(\frac{W}{\gamma+\varepsilon}, -1, +1\right)\right).
$$

The forward pass of a `BitLinear` layer is then

$$
y = \big( \operatorname{LayerNorm}(x) \big)_{q8} \, W_{\text{tern}} \, \gamma.
$$

### 1.4 Straight-Through Estimator

The round operator has gradient $0$ almost everywhere. STE defines a **surrogate gradient**:

$$
\frac{\partial}{\partial x}\operatorname{round}(x) \;\stackrel{\text{STE}}{=}\; 1
$$

implemented in PyTorch with the identity-trick

```python
y = (x.round() - x).detach() + x
```

which evaluates to `x.round()` in the forward pass and to `x` in the backward pass.


## 2 · Reference implementation — `BitLinear` end-to-end

In [ ]:
import math
import torch
import torch.nn as nn

from src.quantization import BitLinear, absmax_quantize_weight, ste_round


# Quick smoke test: construct a layer, push a tensor through it.
torch.manual_seed(0)
layer = BitLinear(in_features=64, out_features=32)
x = torch.randn(4, 16, 64)
y = layer(x)
print("input shape :", tuple(x.shape))
print("output shape:", tuple(y.shape))
print("output mean :", float(y.mean().item()))
print("output std  :", float(y.std().item()))


## 3 · Sanity checks

In [ ]:
# 3.1 Ternarization yields exactly three unique values.
w_tern, gamma = layer.quantize_weights()
unique = sorted(set(w_tern.unique().tolist()))
print("unique ternary values:", unique)
print("scale gamma =", float(gamma.item()))
assert set(unique).issubset({-1.0, 0.0, 1.0})

# 3.2 STE allows gradients through the rounding operator.
x = torch.randn(2, 8, 64, requires_grad=True)
y = layer(x)
loss = y.pow(2).mean()
loss.backward()
print("\nweight.grad has NaN/Inf?  ",
      bool(torch.isnan(layer.weight.grad).any() or torch.isinf(layer.weight.grad).any()))
print("weight.grad max abs:        ", float(layer.weight.grad.abs().max().item()))
print("input.grad max abs:         ", float(x.grad.abs().max().item()))


## 4 · Visualizing the ternarization rule

We sweep a 1-D weight grid through the BitNet ternarization rule and overlay the result with the underlying scale $\gamma$.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def bitnet_ternarize(w: torch.Tensor) -> tuple[torch.Tensor, float]:
    gamma = w.abs().mean()
    return torch.round((w / (gamma + 1e-5)).clamp(-1, 1)), float(gamma)

# Reference weights: Gaussian distribution (as at init).
w = torch.randn(2000) * 0.5
w_tern, gamma = bitnet_ternarize(w)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(w.numpy(), bins=60, alpha=0.7, color="#1f77b4")
axes[0].axvline(+gamma, color="red",   linestyle="--", label=f"+γ = {gamma:.3f}")
axes[0].axvline(-gamma, color="red",   linestyle="--", label=f"-γ = {-gamma:.3f}")
axes[0].set_title("Original Gaussian weight distribution")
axes[0].set_xlabel("weight value");  axes[0].set_ylabel("count");  axes[0].legend()

axes[1].scatter(w.numpy(), w_tern.numpy(), s=4, alpha=0.5, color="#2ca02c")
axes[1].set_title("Ternarization mapping")
axes[1].set_xlabel("original weight"); axes[1].set_ylabel("ternary value")
axes[1].set_yticks([-1, 0, 1]); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nfraction set to  0 : {float((w_tern == 0).float().mean()):.3f}")
print(f"fraction set to ±1 : {float((w_tern.abs() == 1).float().mean()):.3f}")


## 5 · Optimization — measuring the storage benefit

A weight encoded in $\log_2 3 \approx 1.585$ bits compresses by roughly $20\times$ relative to FP32. With $1.6\times$–$2.0\times$ throughput gains reported on dedicated hardware (Ma et al., 2024), this is the headline economic argument for ternary LLMs.


In [ ]:
from math import log2

def storage_bytes(n_weights: int, bits_per_weight: float) -> int:
    return int(round(n_weights * bits_per_weight / 8))

sizes = [64, 128, 256, 512, 1024, 2048, 4096]
rows = []
for d in sizes:
    n = d * d
    fp32_b = storage_bytes(n, 32)
    fp16_b = storage_bytes(n, 16)
    int8_b = storage_bytes(n, 8)
    tern_b = storage_bytes(n, log2(3))   # 1.585 bits / weight
    rows.append((d, fp32_b, fp16_b, int8_b, tern_b,
                 fp32_b / tern_b, fp16_b / tern_b))

print(f"{'d':>5}  {'fp32 (KB)':>10}  {'fp16 (KB)':>10}  {'int8 (KB)':>10}  "
      f"{'tern (KB)':>10}  {'fp32/tern':>10}  {'fp16/tern':>10}")
for d, fp32, fp16, i8, t, r32, r16 in rows:
    print(f"{d:>5}  {fp32/1024:>10.1f}  {fp16/1024:>10.1f}  {i8/1024:>10.1f}  "
          f"{t/1024:>10.1f}  {r32:>10.1f}×  {r16:>10.1f}×")


## 6 · Empirical benchmark — BitLinear vs FP linear on a toy regression

To make the compression argument concrete: we train a single linear layer to fit a small synthetic regression problem in three precisions and read off the final loss.


In [ ]:
import json
from pathlib import Path

torch.manual_seed(0)

# Synthetic linear-regression target  y = X W_true
N, in_dim, out_dim = 1024, 128, 64
W_true = torch.randn(out_dim, in_dim) * 0.1
X = torch.randn(N, in_dim)
Y = X @ W_true.T + 0.01 * torch.randn(N, out_dim)

X_train, X_val = X[:768], X[768:]
Y_train, Y_val = Y[:768], Y[768:]


def train_layer(layer: nn.Module, steps: int = 800, lr: float = 1e-2) -> list[float]:
    opt = torch.optim.AdamW(layer.parameters(), lr=lr)
    history = []
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        pred = layer(X_train)
        loss = (pred - Y_train).pow(2).mean()
        loss.backward()
        opt.step()
        if step % 25 == 0:
            with torch.no_grad():
                history.append(float((layer(X_val) - Y_val).pow(2).mean().item()))
    return history


configs = {
    "FP32 nn.Linear":     nn.Linear(in_dim, out_dim, bias=False),
    "BitLinear (b1.58)":  BitLinear(in_dim, out_dim, bias=False),
}

results = {}
for name, layer in configs.items():
    set_seed(42)
    results[name] = train_layer(layer)
    print(f"{name:25s}  final val MSE = {results[name][-1]:.5f}")


# Save raw numbers
out_path = Path("../benchmarks/results/bitlinear_toy_regression.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({k: v for k, v in results.items()}, indent=2))
print(f"\nresults saved to {out_path}")


# Plot
fig, ax = plt.subplots(figsize=(7, 4))
for name, hist in results.items():
    ax.plot(hist, label=name, linewidth=2)
ax.set_xlabel("evaluation checkpoint (every 25 steps)")
ax.set_ylabel("validation MSE")
ax.set_title("FP32 vs Ternary on a 128→64 linear regression")
ax.set_yscale("log"); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


## 7 · Discussion

- **Gap, not chasm.** On this small problem the ternary layer trains to within a small constant factor of the FP32 baseline. Ma et al. (2024) report that on real LLMs the gap closes entirely above ≈3B parameters.
- **STE caveats.** The straight-through estimator is biased: in regions where the underlying weight is far from the nearest grid point the surrogate gradient *overestimates* progress. Mitigations include weight clipping (already applied here), gradient scaling, or stochastic rounding.
- **Activation quantization matters more than you'd think.** Per-token absmax (as implemented in `src/quantization/absmax.py`) is essential — per-tensor scales destroy accuracy on transformer activations because of outlier tokens (see SmoothQuant).
- **Hardware realization.** The wins materialize only on hardware that exploits the ternary structure: bit-packed memory, addition-only matmuls. On stock CUDA cores you pay the dequantization cost without recouping it.

### References
- Ma, S. et al. "The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits." arXiv:2402.17764, 2024.
- Bengio, Y. et al. "Estimating or Propagating Gradients Through Stochastic Neurons." arXiv:1308.3432, 2013.
- Xiao, G. et al. "SmoothQuant: Accurate and Efficient Post-Training Quantization for LLMs." ICML 2023.
